[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigerUnderTheMoon/CF/blob/main/examples/03_interpreting_audit.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/03_interpreting_audit.ipynb)

Raw notebook URL: [https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/03_interpreting_audit.ipynb](https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/03_interpreting_audit.ipynb)

# Interpreting Audit Quick Example

This notebook connects to `audit.db` (SQLite audit database, SQLite 审计数据库) and queries v2.1 failure events (v2.1 失败事件). It then visualizes the cost distribution of those failures.

Key terminology:
- **AuditLogger**: FMA's structured audit component that persists events to SQLite and JSONL.
- **FailureAudit**: a subtype of audit event recording failure codes, abandonment reasons, and cost in USD.
- **PILOT_BLOCKED**: a governance status meaning pilot evidence exists but readiness gates are not satisfied, so claims cannot be upgraded.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/TigerUnderTheMoon/CF.git"
ROOT = Path.cwd()

if not (ROOT / "README.md").exists() or not (ROOT / "src" / "fma").exists():
    clone_dir = ROOT / "CF"
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    ROOT = clone_dir

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")

## 1. Connect to (or build) the audit database

If `outputs/audit.db` exists, we connect directly. Otherwise, we build a small demo database with v2.1-style failure events so the notebook always runs.

In [ ]:
from datetime import datetime, timezone
from fma.pilot.audit import AuditLogger, AuditEvent, FailureAudit

DB_PATH = ROOT / "outputs" / "audit.db"
AUDIT_DIR = ROOT / "outputs" / "s_fma_v2_1_fresh_holdout"

# Ensure a database exists with some v2.1 demo events
if not DB_PATH.exists():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    logger = AuditLogger(db_path=DB_PATH)
    # Seed a PASS gate
    logger.log_event(
        AuditEvent(
            timestamp=datetime(2026, 6, 6, 10, 0, tzinfo=timezone.utc),
            route_id="v2.1",
            stage="pilot",
            status="PASS",
            event_type="gate",
            message="Pilot gate passed.",
            metadata={},
        )
    )
    # Seed two FAIL events with costs
    logger.log_failure(
        FailureAudit(
            timestamp=datetime(2026, 6, 6, 11, 0, tzinfo=timezone.utc),
            route_id="v2.1",
            stage="full_validation",
            status="FAIL",
            event_type="failure",
            message="Full stochastic validation failed preregistered gates.",
            metadata={"GLOBAL_pass": False},
            failure_codes=["V2_1_FULL_STOCHASTIC_FAIL_SCHEMA_OR_TAGS", "V2_1_FULL_STOCHASTIC_FAIL_SPARSE_SIGNAL"],
            cost_usd=65.689985,
        )
    )
    logger.log_failure(
        FailureAudit(
            timestamp=datetime(2026, 6, 6, 12, 0, tzinfo=timezone.utc),
            route_id="v2.1",
            stage="retry",
            status="FAIL",
            event_type="abandonment",
            message="Strict route abandoned.",
            metadata={},
            failure_codes=["SCHEMA_FAIL"],
            abandonment_reason="transport_unresolved",
            cost_usd=14.0,
        )
    )
    print("Built demo audit.db with v2.1 events.")
else:
    print("Using existing outputs/audit.db.")

logger = AuditLogger(db_path=DB_PATH)
print(f"Connected to {DB_PATH}")

## 2. Query v2.1 failure events

We list all events for route `v2.1` and filter to those with `status == FAIL`. `FailureAudit` records include `failure_codes` and `cost_usd` (美元成本).

In [ ]:
import pandas as pd

events = logger.list_events(route_id="v2.1")
failures = [e for e in events if e.get("status") == "FAIL"]

df = pd.DataFrame(failures)
print(f"Total v2.1 events: {len(events)}")
print(f"v2.1 failure events: {len(failures)}")
df[["timestamp", "stage", "status", "event_type", "message", "failure_codes", "cost_usd"]]

## 3. Visualize cost distribution

A horizontal bar chart shows the cost (in USD) of each v2.1 failure event.

In [ ]:
import matplotlib.pyplot as plt

plot_data = df.sort_values("cost_usd", ascending=True) if not df.empty else pd.DataFrame()
if not plot_data.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(plot_data["stage"], plot_data["cost_usd"].astype(float), color="#E45756")
    ax.set_xlabel("Cost used or projected (USD) 成本（美元）")
    ax.set_ylabel("Stage")
    ax.set_title("v2.1 audit cost distribution\n（v2.1 审计成本分布）")
    fig.tight_layout()
    plt.show()
else:
    print("No failure events to plot.")

## 4. Route summary

`get_route_summary` aggregates failure counts, failure rates, average costs, and the most common failure codes for a given route.

In [ ]:
summary = logger.get_route_summary("v2.1")

print(f"Route: {summary['route_id']}")
print(f"Total events: {summary['total_events']}")
print(f"Failure rate: {summary['failure_rate']:.3f}")
print(f"Average failure cost USD: {summary['average_cost_usd']:.3f}")
print(f"Most common failure codes:")
for item in summary["most_common_failure_codes"]:
    print(f"  - {item['failure_code']}: {item['count']}")

# Render the built-in markdown report for the route
report_md = logger.render_markdown_report("v2.1")
print("\n--- Markdown Report Preview (first 30 lines) ---\n")
print("\n".join(report_md.splitlines()[:30]))

## Interpretation

A **failure event** in this audit table means the artifact should remain bounded by its recorded claim scope. A positive pilot signal or bounded cost does **not** override `PILOT_BLOCKED` (pilot evidence exists but pass gates are not satisfied, 试点证据存在但升级门限未满足) when full validation, sparse-signal, transport, or downstream filtering gates fail.

To return to local attribution computation, see [`01_counterfactual_attribution.ipynb`](01_counterfactual_attribution.ipynb).